# 04 - SARIMA Baseline Model
**Enhancing E-Commerce Sales Forecasting Using Google Trends**

---

## Objectives
1. Build a SARIMA (Seasonal ARIMA) baseline model
2. Forecast sales WITHOUT Google Trends
3. Evaluate performance (RMSE, MAE, MAPE)
4. This serves as the **baseline** to compare against XGBoost + Trends

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

df = pd.read_csv('../data/processed/merged_sales_trends.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f'Loaded: {df.shape}')
print(f'Categories: {df["Category"].unique()}')

## 4.1 Prepare Time Series (Single Category)

SARIMA works on a single univariate time series, so we pick one category.

In [ ]:
# Use the largest category
CATEGORY = df.groupby('Category')['Units_Sold'].sum().idxmax()
print(f'Selected category: {CATEGORY}')

ts = df[df['Category'] == CATEGORY][['Date', 'Units_Sold']].sort_values('Date')
ts = ts.set_index('Date')
ts.index.freq = 'W-SUN'

# Train/test split
SPLIT_DATE = '2024-01-01'
train_ts = ts[ts.index < SPLIT_DATE]
test_ts  = ts[ts.index >= SPLIT_DATE]

print(f'Train: {len(train_ts)} weeks  ({train_ts.index.min()} to {train_ts.index.max()})')
print(f'Test:  {len(test_ts)} weeks  ({test_ts.index.min()} to {test_ts.index.max()})')

ts.plot(figsize=(14, 4), title=f'Weekly Sales: {CATEGORY}', linewidth=1.5)
plt.axvline(pd.Timestamp(SPLIT_DATE), color='red', linestyle='--', label='Train/Test Split')
plt.legend()
plt.show()

## 4.2 SARIMA Model Fitting

SARIMA(p,d,q)(P,D,Q,s) where s=52 (weekly seasonality, annual cycle)

In [ ]:
# SARIMA with seasonal period = 52 weeks
# Using reasonable parameters: (1,1,1)(1,1,1,52)
# For a college project, manual parameter selection is acceptable

print('Fitting SARIMA(1,1,1)(1,1,1,52) ... (this may take a minute)')

model = SARIMAX(
    train_ts['Units_Sold'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False, maxiter=200)
print('Model fitted!')
print(f'\nAIC: {results.aic:.1f}')
print(f'BIC: {results.bic:.1f}')
print(results.summary().tables[1])

## 4.3 Forecast & Evaluate

In [ ]:
# Forecast the test period
n_forecast = len(test_ts)
forecast = results.forecast(steps=n_forecast)

# Align index
forecast.index = test_ts.index

# Metrics
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

rmse = np.sqrt(mean_squared_error(test_ts['Units_Sold'], forecast))
mae  = mean_absolute_error(test_ts['Units_Sold'], forecast)
mape_val = mape(test_ts['Units_Sold'].values, forecast.values)

print('='*50)
print('SARIMA BASELINE RESULTS')
print('='*50)
print(f'RMSE:  {rmse:.2f}')
print(f'MAE:   {mae:.2f}')
print(f'MAPE:  {mape_val:.2f}%')
print('='*50)

In [ ]:
# Plot actual vs forecast
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_ts.index, train_ts['Units_Sold'], color='tab:blue', label='Train', linewidth=1.5)
ax.plot(test_ts.index, test_ts['Units_Sold'], color='tab:green', label='Actual (Test)', linewidth=2)
ax.plot(forecast.index, forecast, color='tab:red', label='SARIMA Forecast', linewidth=2, linestyle='--')

ax.axvline(pd.Timestamp(SPLIT_DATE), color='gray', linestyle=':', alpha=0.7)
ax.fill_between(forecast.index, forecast * 0.85, forecast * 1.15, color='red', alpha=0.1, label='~15% CI')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Units Sold', fontsize=12)
ax.set_title(f'SARIMA Forecast vs Actual - {CATEGORY}\nRMSE={rmse:.1f}, MAE={mae:.1f}, MAPE={mape_val:.1f}%', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4.4 Save Baseline Results

In [ ]:
import json

sarima_results = {
    'model': 'SARIMA(1,1,1)(1,1,1,52)',
    'category': CATEGORY,
    'train_weeks': len(train_ts),
    'test_weeks': len(test_ts),
    'RMSE': round(rmse, 2),
    'MAE': round(mae, 2),
    'MAPE': round(mape_val, 2),
    'AIC': round(results.aic, 1),
    'features_used': 'Units_Sold only (no Google Trends)',
    'note': 'Baseline model for comparison'
}

with open('../data/processed/sarima_results.json', 'w') as f:
    json.dump(sarima_results, f, indent=2)

# Save forecasts
forecast_df = pd.DataFrame({
    'Date': test_ts.index,
    'Actual': test_ts['Units_Sold'].values,
    'SARIMA_Forecast': forecast.values
})
forecast_df.to_csv('../data/processed/sarima_forecasts.csv', index=False)

print('Saved: sarima_results.json, sarima_forecasts.csv')
print(f'\nNext: Compare with XGBoost + Google Trends in Notebook 05')

---

## Summary

| Metric | SARIMA Baseline |
|---|---|
| RMSE | See above |
| MAE | See above |
| MAPE | See above |
| Features | Sales only (univariate) |

**Limitation:** SARIMA only uses historical sales patterns. It cannot incorporate
external signals like Google Trends search interest.

**Next:** `05_xgboost_with_trends.ipynb` - XGBoost model WITH Google Trends

---